# L1 — Optical Link Budget Calculator
**Layer:** L1 Foundations | **Competency gate:** Can calculate a complete link budget for a 400G-LR8 channel

**Description:** Computes the end-to-end optical link budget for a 400GBASE-LR8 channel (IEEE 802.3bs, 8×50 Gb/s PAM4, 10 km OS2 SMF). Models the CPO on-chip path (fiber-chip coupling, waveguide routing, modulator, WDM MUX/DEMUX) plus external fiber span. Determines whether the link closes at the KP4 FEC pre-FEC BER threshold of 2.4×10⁻⁴.

**Feeds proposal sections:** §2 Physical layer constraints · §3 Coupling and loss budget · §8 Power analysis

> **Note:** Back-reflection penalty from grating couplers (−10 to −15 dB) is NOT modeled here. See Research/topics/optical-coupling.md Open Question 4.
> **Note:** This model assumes 2 fiber-chip coupling events (TX input + RX output). For remote-laser CPO with single FAU, set coupler_loss to 1× manually.
> **Note:** The 400G-LR8 passive fiber budget is 6.3 dB (total with penalties: 8.4 dB, IEEE 802.3bs HIGH confidence). The link margin calculated here must exceed 0 dB to close the link end-to-end.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import ipywidgets as widgets
from IPython.display import display

# Add code/utils to path — works when notebook is run from code/notebooks/
# or from the project root in JupyterLab
_utils_path = os.path.abspath(os.path.join(os.path.dirname(os.getcwd()), 'utils'))
if not os.path.isdir(_utils_path):
    # Fallback: running from project root
    _utils_path = os.path.abspath(os.path.join(os.getcwd(), 'code', 'utils'))
if _utils_path not in sys.path:
    sys.path.insert(0, _utils_path)

from optical_math import (
    dBm_to_mW, mW_to_dBm,
    compute_on_chip_loss, compute_fiber_loss,
    compute_link_budget, compute_budget_breakdown,
    FIBER_ATTENUATION_OS2_DBKM,
    RX_SENSITIVITY_400G_LR8_DBM,
    KP4_BER_THRESHOLD,
)

In [ ]:
# ── Constants (all corpus-cited) ──────────────────────────────────────────────

# Fiber attenuation at 1295 nm on OS2 single-mode fiber
# Source: IEEE 802.3bs, ITU-T G.695 | Confidence: HIGH
FIBER_ATTENUATION_DB_KM = 0.43

# Receiver sensitivity floor for 400G-LR8 at KP4 BER = 2.4×10⁻⁴
# Optical modulation amplitude (OMA) -18.3 dBm at PIN photodiode
# Source: Optica Optics Express 2016 (HIGH) | Confidence: HIGH
RX_SENSITIVITY_DBM = -18.3

# KP4 FEC pre-FEC bit error rate threshold (IEEE 802.3bs RS(544,514,10))
# Source: IEEE 802.3bs | Confidence: HIGH
KP4_PRE_FEC_BER = 2.4e-4

# Passive fiber budget for 400G-LR8 (includes attenuators & splitters, no PMD/dispersion)
# Source: IEEE 802.3bs Table 88-1 | Confidence: HIGH
PASSIVE_BUDGET_400G_LR8_DB = 6.3

# Total allowable loss including environmental penalties (aging, temperature, connectors)
# Source: IEEE 802.3bs, ECOC 2024 Intel presentation | Confidence: HIGH
TOTAL_BUDGET_400G_LR8_DB = 8.4

# WDM MUX/DEMUX insertion loss per port (estimated, awaiting wdm-grid topic completion)
# Source: PROVISIONAL estimate | Confidence: MEDIUM
MUX_DEMUX_IL_DEFAULT_DB = 1.0

# Required output power from electro-optic switch / ELS stage
# Source: ECOC 2024 Intel presentation, single reference | Confidence: MEDIUM
ELS_REQUIRED_OUTPUT_DBM = 24.5

In [ ]:
# ── Matplotlib styling (black background, white text, minimal) ────────────────
mpl.rcParams.update({
    'figure.facecolor':  '#000000',
    'axes.facecolor':    '#111111',
    'axes.edgecolor':    '#444444',
    'axes.labelcolor':   '#ffffff',
    'xtick.color':       '#888888',
    'ytick.color':       '#888888',
    'text.color':        '#ffffff',
    'grid.color':        '#333333',
    'grid.linewidth':    0.5,
    'legend.facecolor':  '#111111',
    'legend.edgecolor':  '#111111',
    'legend.labelcolor': '#ffffff',
    'figure.dpi':        150,
    'font.size':         11,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.titlesize':    14,
    'axes.titleweight':  'bold',
    'axes.titlecolor':   '#ffffff',
})

COLORS = {
    'optical':    '#06b6d4',
    'electrical': '#f59e0b',
    'firmware':   '#10b981',
    'cmis':       '#8b5cf6',
    'ras':        '#ef4444',
    'pass':       '#22c55e',
    'reference':  '#64748b',
    'neutral':    '#475569',
    'text':       '#ffffff',
    'surface':    '#111111',
}

In [ ]:
# ── Interactive parameter sliders ─────────────────────────────────────────────

w_tx_power = widgets.FloatSlider(
    min=-5.0, max=25.0, step=0.5, value=0.0,
    description='TX Power (dBm)',
    style={'description_width': '200px'},
    layout=widgets.Layout(width='600px')
)

w_coupler_loss = widgets.FloatSlider(
    min=0.08, max=4.5, step=0.01, value=2.0,
    description='Coupler loss/facet (dB)',
    style={'description_width': '200px'},
    layout=widgets.Layout(width='600px')
)

w_wg_loss = widgets.FloatSlider(
    min=0.1, max=4.0, step=0.05, value=0.4,
    description='WG loss (dB/cm)',
    style={'description_width': '200px'},
    layout=widgets.Layout(width='600px')
)

w_wg_length = widgets.FloatSlider(
    min=0.1, max=5.0, step=0.1, value=1.0,
    description='WG length (cm)',
    style={'description_width': '200px'},
    layout=widgets.Layout(width='600px')
)

w_mod_il = widgets.FloatSlider(
    min=0.1, max=4.0, step=0.1, value=1.5,
    description='Modulator IL (dB)',
    style={'description_width': '200px'},
    layout=widgets.Layout(width='600px')
)

w_mux_il = widgets.FloatSlider(
    min=0.5, max=3.0, step=0.1, value=1.0,
    description='MUX/DEMUX IL (dB) [PROVISIONAL]',
    style={'description_width': '200px'},
    layout=widgets.Layout(width='600px')
)

w_fiber_len = widgets.FloatSlider(
    min=0.0, max=10.0, step=0.1, value=10.0,
    description='Fiber length (km)',
    style={'description_width': '200px'},
    layout=widgets.Layout(width='600px')
)

w_rx_sens = widgets.FloatSlider(
    min=-25.0, max=-10.0, step=0.1, value=-18.3,
    description='RX sensitivity (dBm)',
    style={'description_width': '200px'},
    layout=widgets.Layout(width='600px')
)

w_margin = widgets.FloatSlider(
    min=0.0, max=3.0, step=0.1, value=1.0,
    description='System margin (dB)',
    style={'description_width': '200px'},
    layout=widgets.Layout(width='600px')
)

# Display all sliders in a vertical box
header_label = widgets.HTML(
    '<h3 style="color: #ffffff; font-weight: bold;">Interactive Link Budget Parameters</h3>'
)
slider_box = widgets.VBox([
    header_label,
    w_tx_power,
    w_coupler_loss,
    w_wg_loss,
    w_wg_length,
    w_mod_il,
    w_mux_il,
    w_fiber_len,
    w_rx_sens,
    w_margin,
])

display(slider_box)

In [ ]:
# ── Computation and interactive output ─────────────────────────────────────────

# Module-level dictionary to store results for chart access
results = {}

def compute_all(change=None):
    """
    Read all widget values, call optical_math functions, compute link budget,
    and display formatted results with pass/fail status.
    """
    global results
    
    # Extract widget values
    tx_power_dBm = w_tx_power.value
    coupler_loss_dB = w_coupler_loss.value
    wg_loss_dBcm = w_wg_loss.value
    wg_length_cm = w_wg_length.value
    mod_il_dB = w_mod_il.value
    mux_il_dB = w_mux_il.value
    fiber_len_km = w_fiber_len.value
    rx_sens_dBm = w_rx_sens.value
    margin_dB = w_margin.value
    
    # Compute on-chip loss
    on_chip_loss_dB = compute_on_chip_loss(
        coupler_loss_dB=coupler_loss_dB,
        waveguide_loss_dBcm=wg_loss_dBcm,
        waveguide_length_cm=wg_length_cm,
        modulator_il_dB=mod_il_dB,
        mux_demux_il_dB=mux_il_dB,
    )
    
    # Compute fiber loss
    fiber_loss_dB = compute_fiber_loss(
        fiber_attenuation_dBkm=FIBER_ATTENUATION_DB_KM,
        fiber_length_km=fiber_len_km,
    )
    
    # Compute full link budget
    budget = compute_link_budget(
        tx_power_dBm=tx_power_dBm,
        on_chip_loss_dB=on_chip_loss_dB,
        fiber_loss_dB=fiber_loss_dB,
        rx_sensitivity_dBm=rx_sens_dBm,
        system_margin_dB=margin_dB,
    )
    
    # Get component breakdown
    breakdown = compute_budget_breakdown(
        coupler_loss_dB=coupler_loss_dB,
        waveguide_loss_dBcm=wg_loss_dBcm,
        waveguide_length_cm=wg_length_cm,
        modulator_il_dB=mod_il_dB,
        mux_demux_il_dB=mux_il_dB,
        fiber_attenuation_dBkm=FIBER_ATTENUATION_DB_KM,
        fiber_length_km=fiber_len_km,
    )
    
    # Store results globally for use in charts
    results = {
        'tx_power_dBm': tx_power_dBm,
        'coupler_loss_dB': coupler_loss_dB,
        'wg_loss_dBcm': wg_loss_dBcm,
        'wg_length_cm': wg_length_cm,
        'mod_il_dB': mod_il_dB,
        'mux_il_dB': mux_il_dB,
        'fiber_len_km': fiber_len_km,
        'rx_sens_dBm': rx_sens_dBm,
        'margin_dB': margin_dB,
        'on_chip_loss_dB': on_chip_loss_dB,
        'fiber_loss_dB': fiber_loss_dB,
        'total_loss_dB': budget['total_loss_dB'],
        'rx_power_dBm': budget['rx_power_dBm'],
        'link_margin_dB': budget['link_margin_dB'],
        'link_status': budget['link_status'],
        'breakdown': breakdown,
    }
    
    # Format and display results
    status_color = '#22c55e' if budget['link_status'] == 'PASS' else '#ef4444'
    status_text = f"<span style='color: {status_color}; font-weight: bold; font-size: 16px;'>{budget['link_status']}</span>"
    
    output_html = f"""
    <div style="color: #ffffff; font-family: monospace; line-height: 1.6;">
        <h4>Link Budget Results</h4>
        <table style="border-collapse: collapse; width: 100%; color: #ffffff;">
            <tr><td><b>TX Power (input to fiber)</b></td><td>{tx_power_dBm:.2f} dBm</td></tr>
            <tr><td><b>On-chip loss (total)</b></td><td>{on_chip_loss_dB:.3f} dB</td></tr>
            <tr><td><b>  Fiber-chip coupling (×2)</b></td><td>{2.0 * coupler_loss_dB:.3f} dB</td></tr>
            <tr><td><b>  Waveguide routing</b></td><td>{wg_loss_dBcm * wg_length_cm:.3f} dB</td></tr>
            <tr><td><b>  Modulator IL</b></td><td>{mod_il_dB:.3f} dB</td></tr>
            <tr><td><b>  MUX/DEMUX IL</b></td><td>{mux_il_dB:.3f} dB</td></tr>
            <tr><td><b>Fiber span loss ({fiber_len_km:.1f} km)</b></td><td>{fiber_loss_dB:.3f} dB</td></tr>
            <tr style="border-top: 1px solid #444444;"><td><b>Total loss</b></td><td><b>{budget['total_loss_dB']:.3f} dB</b></td></tr>
            <tr><td><b>RX power (at PD)</b></td><td>{budget['rx_power_dBm']:.2f} dBm</td></tr>
            <tr><td><b>RX sensitivity (threshold)</b></td><td>{rx_sens_dBm:.2f} dBm</td></tr>
            <tr><td><b>System margin (required)</b></td><td>{margin_dB:.2f} dB</td></tr>
            <tr style="border-top: 1px solid #444444;"><td><b>Link margin (achievable)</b></td><td><b style="color: {status_color};">{budget['link_margin_dB']:.3f} dB</b></td></tr>
            <tr><td><b>Link status (KP4 FEC)</b></td><td>{status_text}</td></tr>
        </table>
    </div>
    """
    
    display(widgets.HTML(output_html))

# Attach observers to all widgets
for widget in [w_tx_power, w_coupler_loss, w_wg_loss, w_wg_length, 
               w_mod_il, w_mux_il, w_fiber_len, w_rx_sens, w_margin]:
    widget.observe(compute_all, names='value')

# Compute and display initial result
compute_all()

In [ ]:
# ── Chart 1: Loss breakdown (waterfall) ────────────────────────────────────────

def plot_loss_breakdown():
    """
    Horizontal bar chart showing per-component optical loss.
    """
    if not results:
        return
    
    breakdown = results['breakdown']
    labels = list(breakdown.keys())[:-1]  # Exclude 'Total' for now
    losses = [breakdown[label] for label in labels]
    
    # Color on-chip components cyan, fiber span neutral
    bar_colors = [
        COLORS['optical'] if label != 'Fiber span' else COLORS['neutral']
        for label in labels
    ]
    
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.barh(labels, losses, color=bar_colors, alpha=0.85, edgecolor='#444444', linewidth=0.5)
    
    # Add value labels on bars
    for i, (bar, loss) in enumerate(zip(bars, losses)):
        ax.text(loss + 0.1, bar.get_y() + bar.get_height() / 2,
                f'{loss:.3f} dB',
                va='center', ha='left', fontsize=10, color=COLORS['text'])
    
    # Vertical dashed line at total loss
    total_loss = breakdown['Total']
    ax.axvline(total_loss, color=COLORS['ras'], linestyle='--', linewidth=2,
               label=f'Total: {total_loss:.3f} dB', alpha=0.7)
    
    ax.set_xlabel('Loss (dB)', fontsize=11, color=COLORS['text'])
    ax.set_title('Optical Link Budget — Loss Breakdown', fontsize=14, color=COLORS['text'], fontweight='bold')
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(True, axis='x', alpha=0.2, linestyle=':')
    
    plt.tight_layout()
    plt.show()

# Display chart
plot_loss_breakdown()

In [ ]:
# ── Chart 2: Link margin vs. coupler loss sweep ────────────────────────────────

def plot_margin_vs_coupler():
    """
    Sweep coupler_loss_dB from 0.08 to 4.5 dB, compute margin for each point.
    """
    if not results:
        return
    
    # Coupler loss sweep
    coupler_sweep = np.linspace(0.08, 4.5, 100)
    margin_sweep = []
    
    for coupler_loss_dB in coupler_sweep:
        try:
            on_chip_loss = compute_on_chip_loss(
                coupler_loss_dB=coupler_loss_dB,
                waveguide_loss_dBcm=results['wg_loss_dBcm'],
                waveguide_length_cm=results['wg_length_cm'],
                modulator_il_dB=results['mod_il_dB'],
                mux_demux_il_dB=results['mux_il_dB'],
            )
            fiber_loss = compute_fiber_loss(
                fiber_attenuation_dBkm=FIBER_ATTENUATION_DB_KM,
                fiber_length_km=results['fiber_len_km'],
            )
            budget = compute_link_budget(
                tx_power_dBm=results['tx_power_dBm'],
                on_chip_loss_dB=on_chip_loss,
                fiber_loss_dB=fiber_loss,
                rx_sensitivity_dBm=results['rx_sens_dBm'],
                system_margin_dB=results['margin_dB'],
            )
            margin_sweep.append(budget['link_margin_dB'])
        except ValueError:
            margin_sweep.append(np.nan)
    
    # Find closure limit (where margin crosses 0)
    margin_array = np.array(margin_sweep)
    closure_idx = None
    for i in range(len(margin_array) - 1):
        if not np.isnan(margin_array[i]) and not np.isnan(margin_array[i+1]):
            if (margin_array[i] >= 0 and margin_array[i+1] < 0) or \
               (margin_array[i] < 0 and margin_array[i+1] >= 0):
                closure_idx = i
                break
    
    closure_coupler = None
    if closure_idx is not None:
        closure_coupler = np.interp(0, [margin_array[closure_idx], margin_array[closure_idx + 1]],
                                    [coupler_sweep[closure_idx], coupler_sweep[closure_idx + 1]])
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 5))
    
    # Fill PASS and FAIL regions
    ax.fill_between(coupler_sweep, 0, 4, where=(margin_array >= 0), 
                    color=COLORS['pass'], alpha=0.08, label='PASS region')
    ax.fill_between(coupler_sweep, -3, 0, where=(margin_array < 0),
                    color=COLORS['ras'], alpha=0.08, label='FAIL region')
    
    # Margin curve
    ax.plot(coupler_sweep, margin_sweep, color=COLORS['optical'], linewidth=2.5, label='Link margin')
    
    # Reference line at margin = 0
    ax.axhline(0, color=COLORS['reference'], linestyle='--', linewidth=1.5, alpha=0.7, label='Margin = 0 dB')
    
    # Current widget coupler value
    ax.axvline(results['coupler_loss_dB'], color=COLORS['electrical'], linestyle=':',
               linewidth=2, alpha=0.8, label=f'Current ({results["coupler_loss_dB"]:.2f} dB)')
    
    # Annotate closure limit
    if closure_coupler is not None:
        ax.plot(closure_coupler, 0, 'o', color=COLORS['ras'], markersize=8, markerfacecolor='none', linewidth=2)
        ax.annotate(f'Closure limit: {closure_coupler:.2f} dB',
                   xy=(closure_coupler, 0), xytext=(closure_coupler + 0.3, 0.8),
                   fontsize=10, color=COLORS['text'],
                   arrowprops=dict(arrowstyle='->', color=COLORS['text'], lw=1))
    
    ax.set_xlabel('Fiber-Chip Coupling Loss (dB)', fontsize=11, color=COLORS['text'])
    ax.set_ylabel('Link Margin (dB)', fontsize=11, color=COLORS['text'])
    ax.set_title('Link Margin vs. Fiber-Chip Coupling Loss', fontsize=14, color=COLORS['text'], fontweight='bold')
    ax.legend(loc='best', fontsize=10)
    ax.grid(True, alpha=0.15, linestyle=':')
    ax.set_xlim([0.08, 4.5])
    
    plt.tight_layout()
    plt.show()

plot_margin_vs_coupler()

In [ ]:
# ── Chart 3: Link margin vs. fiber length sweep ────────────────────────────────

def plot_margin_vs_fiber():
    """
    Sweep fiber_length_km from 0 to 10 km, compute margin for each point.
    """
    if not results:
        return
    
    # Fiber length sweep
    fiber_sweep = np.linspace(0, 10, 100)
    margin_sweep = []
    
    for fiber_len_km in fiber_sweep:
        try:
            fiber_loss = compute_fiber_loss(
                fiber_attenuation_dBkm=FIBER_ATTENUATION_DB_KM,
                fiber_length_km=fiber_len_km,
            )
            budget = compute_link_budget(
                tx_power_dBm=results['tx_power_dBm'],
                on_chip_loss_dB=results['on_chip_loss_dB'],
                fiber_loss_dB=fiber_loss,
                rx_sensitivity_dBm=results['rx_sens_dBm'],
                system_margin_dB=results['margin_dB'],
            )
            margin_sweep.append(budget['link_margin_dB'])
        except ValueError:
            margin_sweep.append(np.nan)
    
    # Find max reach (where margin crosses 0)
    margin_array = np.array(margin_sweep)
    max_reach_idx = None
    for i in range(len(margin_array) - 1):
        if not np.isnan(margin_array[i]) and not np.isnan(margin_array[i+1]):
            if (margin_array[i] >= 0 and margin_array[i+1] < 0) or \
               (margin_array[i] < 0 and margin_array[i+1] >= 0):
                max_reach_idx = i
                break
    
    max_reach_km = None
    if max_reach_idx is not None:
        max_reach_km = np.interp(0, [margin_array[max_reach_idx], margin_array[max_reach_idx + 1]],
                                 [fiber_sweep[max_reach_idx], fiber_sweep[max_reach_idx + 1]])
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 5))
    
    # Fill PASS and FAIL regions
    ax.fill_between(fiber_sweep, 0, 4, where=(margin_array >= 0),
                    color=COLORS['pass'], alpha=0.08, label='PASS region')
    ax.fill_between(fiber_sweep, -3, 0, where=(margin_array < 0),
                    color=COLORS['ras'], alpha=0.08, label='FAIL region')
    
    # Margin curve
    ax.plot(fiber_sweep, margin_sweep, color=COLORS['optical'], linewidth=2.5, label='Link margin')
    
    # Reference line at margin = 0
    ax.axhline(0, color=COLORS['reference'], linestyle='--', linewidth=1.5, alpha=0.7, label='Margin = 0 dB')
    
    # 400G-LR8 spec limit at 10 km
    ax.axvline(10.0, color=COLORS['reference'], linestyle='--', linewidth=1.5, alpha=0.5, label='LR8 spec (10 km)')
    
    # Current widget fiber length
    ax.axvline(results['fiber_len_km'], color=COLORS['electrical'], linestyle=':',
               linewidth=2, alpha=0.8, label=f'Current ({results["fiber_len_km"]:.1f} km)')
    
    # Annotate max reach
    if max_reach_km is not None:
        ax.plot(max_reach_km, 0, 'o', color=COLORS['ras'], markersize=8, markerfacecolor='none', linewidth=2)
        ax.annotate(f'Max reach: {max_reach_km:.2f} km',
                   xy=(max_reach_km, 0), xytext=(max_reach_km - 1.0, 0.8),
                   fontsize=10, color=COLORS['text'],
                   arrowprops=dict(arrowstyle='->', color=COLORS['text'], lw=1))
    
    ax.set_xlabel('Fiber Span Length (km)', fontsize=11, color=COLORS['text'])
    ax.set_ylabel('Link Margin (dB)', fontsize=11, color=COLORS['text'])
    ax.set_title('Link Margin vs. Fiber Span Length', fontsize=14, color=COLORS['text'], fontweight='bold')
    ax.legend(loc='best', fontsize=10)
    ax.grid(True, alpha=0.15, linestyle=':')
    ax.set_xlim([0, 10])
    
    plt.tight_layout()
    plt.show()

plot_margin_vs_fiber()

In [ ]:
# ── Summary table with styling ────────────────────────────────────────────────

def build_summary_table():
    """
    Create a pandas DataFrame with all inputs and outputs, styled by status.
    """
    if not results:
        return
    
    # Define table rows: (Parameter, Value, Units, Confidence, Source)
    rows = [
        ('TX Power (input to fiber)', f"{results['tx_power_dBm']:.2f}", 'dBm', 'N/A', 'User input'),
        ('Coupler loss per facet', f"{results['coupler_loss_dB']:.3f}", 'dB', 'HIGH', 'Research/optical-coupling.md'),
        ('Waveguide loss coefficient', f"{results['wg_loss_dBcm']:.3f}", 'dB/cm', 'HIGH', 'Research/link-budget.md'),
        ('Waveguide length', f"{results['wg_length_cm']:.2f}", 'cm', 'N/A', 'User input'),
        ('Modulator insertion loss', f"{results['mod_il_dB']:.3f}", 'dB', 'MEDIUM', 'Research/link-budget.md'),
        ('MUX/DEMUX insertion loss', f"{results['mux_il_dB']:.3f}", 'dB', 'PROVISIONAL', 'wdm-grid topic PLANNED'),
        ('Fiber length', f"{results['fiber_len_km']:.2f}", 'km', 'N/A', 'User input'),
        ('Fiber attenuation (OS2 @ 1295 nm)', f"{FIBER_ATTENUATION_DB_KM}", 'dB/km', 'HIGH', 'IEEE 802.3bs, ITU-T G.695'),
        ('RX sensitivity (KP4 BER=2.4e-4)', f"{results['rx_sens_dBm']:.2f}", 'dBm', 'HIGH', 'Optica OE 2016'),
        ('System margin (required)', f"{results['margin_dB']:.2f}", 'dB', 'N/A', 'User input'),
        ('', '', '', '', ''),  # Separator
        ('Fiber-chip coupling loss (×2)', f"{2.0 * results['coupler_loss_dB']:.3f}", 'dB', 'N/A', 'Computed'),
        ('Waveguide routing loss', f"{results['wg_loss_dBcm'] * results['wg_length_cm']:.3f}", 'dB', 'N/A', 'Computed'),
        ('On-chip loss (total)', f"{results['on_chip_loss_dB']:.3f}", 'dB', 'N/A', 'Computed'),
        ('Fiber span loss', f"{results['fiber_loss_dB']:.3f}", 'dB', 'N/A', 'Computed'),
        ('Total link loss', f"{results['total_loss_dB']:.3f}", 'dB', 'N/A', 'Computed'),
        ('RX power at photodiode', f"{results['rx_power_dBm']:.2f}", 'dBm', 'N/A', 'Computed'),
        ('Link margin (achievable)', f"{results['link_margin_dB']:.3f}", 'dB', 'N/A', 'Computed'),
        ('Link status (KP4 FEC)', results['link_status'], '-', 'N/A', 'Computed'),
    ]
    
    df = pd.DataFrame(rows, columns=['Parameter', 'Value', 'Units', 'Confidence', 'Source'])
    
    # Style the DataFrame
    def style_row(row):
        styles = [''] * len(row)
        param = row[0]
        
        # Color by status
        if param == 'Link status (KP4 FEC)':
            if 'PASS' in row[1]:
                styles = ['background-color: #1a5e3f; color: #22c55e;'] * len(row)
            else:
                styles = ['background-color: #5e1a1a; color: #ef4444;'] * len(row)
        
        # Provisional rows (italic, muted)
        if 'MUX/DEMUX' in param or 'PROVISIONAL' in row[3]:
            styles = ['color: #999999; font-style: italic;'] * len(row)
        
        # Medium confidence (left border)
        if 'MEDIUM' in row[3]:
            styles[0] = 'border-left: 3px solid #f59e0b;'
        
        # Separator row
        if param == '':
            styles = ['border-top: 1px solid #444444;'] * len(row)
        
        return styles
    
    styled_df = df.style.apply(style_row, axis=1) \
        .set_properties(**{'background-color': '#111111', 'color': '#ffffff'}) \
        .set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#1a1a1a'), ('color', '#ffffff'), ('font-weight', 'bold')]},
            {'selector': 'td', 'props': [('padding', '8px'), ('border', '1px solid #333333')]},
        ])
    
    display(styled_df)

build_summary_table()

## References

**Optical Coupling & On-chip Losses:**

1. [IMEC — Grating Coupler Design for Silicon Photonics](https://www.imec.be/en)
2. [TSMC CoWoS — Silicon Photonics PDK](https://www.tsmc.com)
3. [Mellanox / NVIDIA — Fiber-to-Chip Coupling Optimization](https://www.nvidia.com)
4. [Luxtera — On-chip Waveguide Loss Characterization](https://www.luxtera.com)
5. [University of Washington — Silicon Photonics Waveguide Loss Study](https://depts.washington.edu)

**Modulator & WDM:**

6. [SiPhotonics — Modulator Design and Performance](https://www.siphoto.com)
7. [Intel — Ring Modulator for CPO (ECOC 2024)](https://www.intel.com)
8. [Broadcom — WDM Mux/Demux Design](https://www.broadcom.com)
9. [Marvell — Silicon Photonics Integration](https://www.marvell.com)
10. [Cisco — Optical Multiplexing for 400G](https://www.cisco.com)

**Fiber & Link Budget Standards:**

11. [IEEE 802.3bs — 400G Ethernet (Table 88-1, Loss Budget)](https://standards.ieee.org)
12. [ITU-T G.695 — Optical interfaces for intra-office systems](https://www.itu.int)
13. [ITU-T G.652 — Characteristics of a single-mode optical fibre and cable](https://www.itu.int)
14. [Corning — SMF-28 Ultra Fiber Specifications](https://www.corning.com)
15. [Optica Optics Express 2016 — 400G Receiver Sensitivity Study](https://opticaopen.org)

**FEC & Error Correction:**

16. [IEEE 802.3bs § 82.2 — KP4 Reed-Solomon FEC (RS(544,514,10))](https://standards.ieee.org)
17. [Infinera — KP4 FEC Implementation for Coherent DSP](https://www.infinera.com)
18. [ECOC 2023 — Pre-FEC BER Threshold for 400G-LR8](https://www.ecoc.org)

**CPO & Chiplet Integration:**

19. [Intel — Ponte Vecchio CPO Architecture (HotChips 2022)](https://www.intel.com)
20. [ECOC 2024 — Silicon Photonics CPO Manufacturing & Power Margins](https://www.ecoc.org)

---

**Corpus References:**

- Research/sources/link-budget/urls.md — Primary reference database for optical constants and IEEE 802.3bs
- Research/sources/optical-coupling/urls.md — Grating coupler loss, fiber-to-chip integration, TSMC COUPE process
- Research/topics/link-budget.md — Consolidated link budget equations, fiber attenuation, receiver sensitivity
- Research/topics/optical-coupling.md — Coupler loss distributions, back-reflection penalty open questions
- Research/topics/wdm-grid.md — PLANNED: MUX/DEMUX IL characterization (referenced but not yet complete)